# Plant Diseases AI — treino no Colab

Classificação de imagens de plantas em 38 classes com YOLO26n-cls.

1. No Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
2. Envie `plant-diseases-colab.tar.gz` para `Meu Drive/plant-diseases-ai/`.
3. Execute as células na ordem. Pesos e resultados serão salvos no Drive.

**Se um treino já estiver rodando em outro notebook, deixe-o terminar.**
Este arquivo permite guardar e reproduzir o procedimento; não precisa iniciar outra execução.
Para avaliar um treino concluído, prepare o ambiente e os dados, ajuste `RUN_NAME` e pule a célula de treino.
As GPUs gratuitas e a duração da sessão dependem dos limites do Colab.


## 1. Dependências

A versão do Ultralytics corresponde ao teste inicial que concluiu com sucesso.
O PyTorch com CUDA é fornecido pelo Colab; as versões efetivas serão registradas com o treino.


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.161"],
    check=True,
)


## 2. Drive e configuração

Use `EPOCHS = 3` e um nome novo para um teste curto. Para o treino completo, mantenha 30 épocas.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/plant-diseases-ai")
PROJECT_DIR = Path("/content/plant-diseases-ai")
DATASET_DIR = PROJECT_DIR / "dataset"
ARCHIVE = DRIVE_DIR / "plant-diseases-colab.tar.gz"
RUN_NAME = "plant-diseases-v1"
EPOCHS = 30
RUN_DIR = DRIVE_DIR / "runs" / RUN_NAME
BEST_PATH = RUN_DIR / "weights" / "best.pt"


## 3. Preparar os dados locais

Leia as imagens no disco do Colab. Mantenha no Drive o arquivo compactado e os resultados.
A extração exige uma pasta de dataset ainda inexistente para não misturar conjuntos de arquivos.
Se esta célula já concluiu nesta sessão, pule-a ao avaliar um modelo.


In [ ]:
import shutil
import tarfile

if not ARCHIVE.is_file():
    raise FileNotFoundError(f"Envie o arquivo para: {ARCHIVE}")
if DATASET_DIR.exists():
    raise FileExistsError(
        "O dataset já existe nesta sessão. Se a extração anterior concluiu, "
        "pule esta célula e confira os dados abaixo. Para trocar os dados, "
        "use um novo ambiente ou uma pasta local nova."
    )

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
local_archive = Path("/content/plant-diseases-colab.tar.gz")
shutil.copy2(ARCHIVE, local_archive)
with tarfile.open(local_archive, "r:gz") as archive:
    archive.extractall(PROJECT_DIR, filter="data")

print("Dados extraídos em:", DATASET_DIR)


## 4. Conferir dados e GPU

Contagens esperadas para o dataset deduplicado deste projeto.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU no ambiente do Colab antes de treinar.")
print("GPU:", torch.cuda.get_device_name(0))

expected_counts = {"train": 43412, "val": 5415, "test": 5457}
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_sets = {}
for split, expected in expected_counts.items():
    split_dir = DATASET_DIR / split
    classes = sorted(path for path in split_dir.iterdir() if path.is_dir())
    class_sets[split] = {path.name for path in classes}
    count = sum(
        1 for folder in classes for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )
    print(f"{split}: {count} imagens em {len(classes)} classes")
    if count != expected or len(classes) != 38:
        raise ValueError(f"Confira a extração e o dataset: contagem inesperada em {split}.")

if not (class_sets["train"] == class_sets["val"] == class_sets["test"]):
    raise ValueError("As classes diferem entre treino, validação e teste.")


## 5. Treinar

O treino seleciona `best.pt` pela validação. O conjunto de teste permanece reservado.
Esta célula inicia uma execução nova; não retoma automaticamente um treino interrompido.
Se a pasta da execução já existir, ela será preservada e a célula não iniciará outro treino.


In [ ]:
import json
import platform
import ultralytics
from ultralytics import YOLO

if RUN_DIR.exists():
    raise FileExistsError(
        f"A execução {RUN_DIR} já existe. Para avaliar o modelo salvo, "
        "pule esta célula. Para um novo experimento, escolha outro RUN_NAME."
    )

model = YOLO("yolo26n-cls.pt")

def record_environment(trainer):
    environment = {
        "python": platform.python_version(),
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    }
    (Path(trainer.save_dir) / "environment.json").write_text(
        json.dumps(environment, indent=2), encoding="utf-8"
    )

model.add_callback("on_train_start", record_environment)
model.train(
    data=str(DATASET_DIR),
    epochs=EPOCHS,
    imgsz=224,
    batch=16,
    patience=10,
    workers=2,
    device=0,
    seed=42,
    project=str(DRIVE_DIR / "runs"),
    name=RUN_NAME,
)
BEST_PATH = Path(model.trainer.best)
print("Melhor modelo:", BEST_PATH)


## 6. Avaliar no teste reservado

Execute depois de concluir o treinamento e escolher o modelo pela validação.
Não use os resultados de teste para escolher hiperparâmetros.
Esta etapa também funciona em uma sessão nova, após preparar ambiente, Drive e dados.


In [ ]:
from datetime import datetime, timezone
from ultralytics import YOLO

if not BEST_PATH.is_file():
    raise FileNotFoundError(f"Modelo não encontrado: {BEST_PATH}")

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
EVAL_DIR = DRIVE_DIR / "evaluations" / f"{RUN_NAME}-test-{stamp}"
best_model = YOLO(str(BEST_PATH))
metrics = best_model.val(
    data=str(DATASET_DIR),
    split="test",
    device=0,
    plots=True,
    project=str(EVAL_DIR.parent),
    name=EVAL_DIR.name,
)
print(f"Acurácia top-1 no teste: {metrics.top1:.2%}")
print(f"Acurácia top-5 no teste: {metrics.top5:.2%}")
print("Resultados:", EVAL_DIR)


## 7. Guardar métricas e baixar o modelo

O `best.pt` já está no Drive. O JSON abaixo é um relatório pequeno que pode ser
versionado junto com a matriz de confusão. Antes de enviar o notebook ao GitHub,
limpe as saídas das células; mantenha imagens e pesos fora do Git.


In [ ]:
import json
from google.colab import files

report = {
    "run_name": RUN_NAME,
    "checkpoint": str(BEST_PATH.relative_to(DRIVE_DIR)),
    "split": "test",
    "top1_accuracy": float(metrics.top1),
    "top5_accuracy": float(metrics.top5),
    "test_images": 5457,
    "classes": 38,
}
report_path = EVAL_DIR / "test_metrics.json"
report_path.write_text(
    json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Relatório salvo:", report_path)
files.download(str(BEST_PATH))
